# W1–W2 · Companies compared inside each country: does the pattern repeat?

> **Question:** Inside one country, all companies face the same economy, tax system and institutions. So differences in ROA *between companies of the same country* cannot come from country conditions. Which company traits go with higher ROA there, and is the same pattern found in country after country?

**New rules for this series (3.x).** The earlier notebooks (0.x to 2.x) compared **countries**, where only 34 data points exist. Here the comparison is **between companies inside a country**, and each country is a **replication**: we run the same analysis in every country and ask whether it repeats.

| | Earlier notebooks (1.x, 2.x) | This series (3.x) |
|---|---|---|
| Unit compared | countries (n = 34) | companies inside a country (thousands) |
| Country conditions | the thing being studied | held fixed (they are shared inside a country) |
| Evidence | one p-value from a wild cluster bootstrap | the same estimate in each country, combined by **random-effects meta-analysis** (country = unit), plus how often the sign repeats |
| Verdict rule | sign + wild p < 0.05 + robust in 80% of samples | predicted sign + random-effects p < 0.05 + **same sign in at least 80% of countries** (`utils.verdict` with that share) |
| Standard errors | clustered by country | clustered by company inside each country |

**Traits (company level, from the Yahoo accounts):**
- **W1, financing: equity ratio** (equity / total assets). *Prediction:* positive; firms funded more by equity carry less debt cost and are sturdier.
- **W2, size: percentile of total assets within the country and year.** *Prediction:* none (two-sided). Large firms have scale advantages, small firms have niches and growth; theory points both ways.
- Both are estimated with sector and fiscal-year controls: `ROA (pp) ~ trait + sector + fiscal year`, one regression per country.

**Honest framing.** Before this notebook was finished I ran a quick check of both traits on this same data, so the results below are not a blind pre-registration; both traits are reported and both enter the multiple-testing correction. Each country's own estimate has only a few hundred companies, so single-country results are noisy and the interesting part is the pattern across countries. A relationship that repeats inside every country is much stronger evidence of a real company-level regularity than any single country-level comparison, but it is still **association, not causation** (an equity ratio can be high because the firm has been profitable: reverse causation).

In [1]:
import sys, pathlib, warnings
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from scipy import stats
import statsmodels.formula.api as smf

warnings.filterwarnings("ignore")
ROOT = pathlib.Path.cwd().resolve()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src" / "analysis"))
import prep, utils, within
from utils import bh

con = prep.connect()
pd.set_option("display.width", 220, "display.max_columns", 40, "display.max_rows", 120, "display.float_format", "{:,.3f}".format)
px.defaults.template = "plotly_white"

country = prep.load_country(con)
dim = country["dim"]
fy = prep.load_firms(con, country)               # Yahoo company-years
names = dim.name
n_firms = fy.groupby("iso3").company_id.nunique()
ok = n_firms[n_firms >= within.MIN_FIRMS].index
fy = fy[fy.iso3.isin(ok)].copy()
print(f"{len(ok)} of {len(dim)} countries have at least {within.MIN_FIRMS} companies and enter the analysis; left out: " + ", ".join(f"{names[c]} ({n_firms.get(c, 0)})" for c in dim.index if c not in ok))
print(f"{len(fy):,} company-years of {fy.company_id.nunique():,} companies")

28 of 34 countries have at least 30 companies and enter the analysis; left out: Czechia (17), Estonia (18), Hungary (21), Latvia (10), Lithuania (22), Luxembourg (10)
9,837 company-years of 2,492 companies


## 1 · Estimates inside each country
One regression per country; the table shows the coefficient (percentage points of ROA) for a **+0.1 change in the equity ratio** (W1) and for **moving from the smallest to the largest company** in the country (W2).

In [2]:
TERMS = {"W1": "equity_ratio_w", "W2": "size_rank"}
slopes = pd.concat([within.country_slopes(fy, [t]) for t in TERMS.values()], ignore_index=True)     # one trait at a time, as stated
slopes["country"] = slopes.iso3.map(names)
res = {h: within.repeats(slopes[slopes.term == t]) for h, t in TERMS.items()}
unit = {"W1": 0.1, "W2": 1.0}      # report W1 per +0.1 of equity ratio
def forest(h, title):
    s = slopes[slopes.term == TERMS[h]].assign(coef=lambda d: d.coef * unit[h], lo=lambda d: (d.coef - 1.96 * d.se * unit[h]), hi=lambda d: (d.coef + 1.96 * d.se * unit[h])).sort_values("coef")
    r = res[h]["re"]; mu = r["mean"] * unit[h]
    fig = px.scatter(s, x="coef", y="country", error_x=s.hi - s.coef, error_x_minus=s.coef - s.lo, title=title)
    fig.add_vline(x=0, line_color="grey"); fig.add_vline(x=mu, line_color="#e53935", line_dash="dot", annotation_text=f"pooled {mu:+.2f}")
    fig.update_layout(height=700, yaxis_title=""); fig.show()
forest("W1", "W1 · ROA change (pp) per +0.1 equity ratio, inside each country")
forest("W2", "W2 · ROA difference (pp), largest vs smallest company, inside each country")

## 2 · Does it repeat?
Random-effects pooling across countries (Hartung-Knapp, suited to a modest number of countries), the share of countries with the pooled sign, and leave-one-country-out agreement (does the pattern of the other countries predict the sign in a country that was left out?).

In [3]:
rows = []
for h, t in TERMS.items():
    r = res[h]; re = r["re"]
    rows.append({"id": h, "trait": t, "unit": "per +0.1" if h == "W1" else "largest vs smallest", "pooled (pp)": re["mean"] * unit[h], "ci_low": re["ci_low"] * unit[h], "ci_high": re["ci_high"] * unit[h], "p (random effects)": re["p"],
                 "countries": re["k"], "same sign": r["same_sign_share"], "significant, same sign": r["significant_same_sign"], "significant, opposite": r["significant_opposite"], "leave-one-out agreement": r["loo_agreement"],
                 "tau (pp)": re["tau"] * unit[h], "I2": re["I2"]})
summary = pd.DataFrame(rows).set_index("id"); summary["q (BH)"] = bh(summary["p (random effects)"]); summary.round(3)

,trait,unit,pooled (pp),ci_low,ci_high,p (random effects),countries,same sign,"significant, same sign","significant, opposite",leave-one-out agreement,tau (pp),I2,q (BH)
id,,,,,,,,,,,,,,
W1,equity_ratio_w,per +0.1,1.159,0.956,1.362,0.000,28,0.964,22,0,0.964,0.385,0.579,0.000
W2,size_rank,largest vs smallest,-0.426,-2.962,2.109,0.733,28,0.500,5,4,0.500,5.085,0.771,0.733


How to read it: **tau** is how much the *true* slope differs from country to country (in pp); **I²** is the share of the differences between country estimates that is real variation and not sampling noise. A trait that repeats has a pooled effect clearly away from zero, a high same-sign share and a tau that is small next to it.

## 3 · The same comparison without holding countries fixed
Pooling all companies with and without country × year fixed effects: how much of the company-level relationship is a country-level one in disguise?

In [4]:
pool = {}
for label, f in [("no country control", "roa_pp ~ equity_ratio_w + size_rank + C(sector) + C(fiscal_year)"),
                 ("country × year fixed effects", "roa_pp ~ equity_ratio_w + size_rank + C(sector) + C(iso3):C(fiscal_year)")]:
    d = fy.dropna(subset=["roa_pp", "equity_ratio_w", "size_rank"])
    m = smf.ols(f, d).fit(cov_type="cluster", cov_kwds={"groups": pd.factorize(d.company_id)[0]})
    pool[label] = {"equity ratio (per +0.1)": m.params["equity_ratio_w"] * 0.1, "se1": m.bse["equity_ratio_w"] * 0.1, "size (largest vs smallest)": m.params["size_rank"], "se2": m.bse["size_rank"], "firm-years": int(m.nobs)}
pd.DataFrame(pool).T.round(3)

,equity ratio (per +0.1),se1,size (largest vs smallest),se2,firm-years
no country control,1.159,0.097,3.999,0.750,"9,837.000"
country × year fixed effects,1.250,0.094,4.190,0.723,"9,837.000"


## 4 · Does the pattern depend on the country? (exploratory)
The country slopes against country development and institutions. Only 28 countries, so this is a lead, not a test.

In [5]:
dev, inst = country["dev_idx"], country["inst_idx"]
rows = []
for h, t in TERMS.items():
    s = slopes[slopes.term == t].set_index("iso3")
    for nm, idx in [("development", dev), ("institutions", inst)]:
        r, p, n = utils.spearman(s.coef, idx.reindex(s.index))
        rows.append({"id": h, "trait": t, "country feature": nm, "rho": r, "p": p, "countries": n})
het = pd.DataFrame(rows).set_index(["id", "country feature"]); het["q (BH)"] = bh(het.p); het.round(3)

trait    rho     p  countries  q (BH)
id country feature                                                
W1 development      equity_ratio_w -0.013 0.947         28   0.947
   institutions     equity_ratio_w -0.089 0.654         28   0.872
W2 development           size_rank  0.343 0.074         28   0.295
   institutions          size_rank  0.246 0.207         28   0.415

In [6]:
import plotly.subplots as ps
fig = ps.make_subplots(rows=1, cols=2, subplot_titles=("W1 equity ratio slope", "W2 size slope"))
for i, (h, t) in enumerate(TERMS.items(), 1):
    s = slopes[slopes.term == t].set_index("iso3")
    fig.add_trace(go.Scatter(x=dev.reindex(s.index), y=s.coef * unit[h], mode="markers+text", text=s.index, textposition="top center", marker_size=8, showlegend=False), row=1, col=i)
    fig.update_xaxes(title_text="development index (SD)", row=1, col=i)
fig.update_yaxes(title_text="slope (pp)", row=1, col=1); fig.update_layout(height=420, title="Country slope vs country development"); fig.show()

## 5 · Sectors: is the pecking order the same in every country?
For each country, each sector's median ROA minus the country median. Then: how well does one country's sector ranking match the average of all other countries? The placebo shuffles the sector labels inside each country, which destroys any real sector effect.

In [7]:
MIN_SEC = 8
def sector_profile(d):
    g = d.groupby(["iso3", "sector"]).roa_pp.agg(["median", "size"]).reset_index()
    g = g[g["size"] >= MIN_SEC]
    g["prem"] = g["median"] - g.iso3.map(d.groupby("iso3").roa_pp.median())
    return g.pivot(index="iso3", columns="sector", values="prem")
def mean_agreement(P):
    out = {}
    for c in P.index:
        other = P.drop(c).mean()
        m = P.loc[c].notna() & other.notna()
        if m.sum() >= 5:
            out[c] = stats.spearmanr(P.loc[c][m], other[m])[0]
    return pd.Series(out)
P = sector_profile(fy)
agree = mean_agreement(P)
rng = np.random.default_rng(0)
plc = []
for _ in range(30):
    d = fy.copy(); d["sector"] = d.groupby("iso3").sector.transform(lambda s: rng.permutation(s.values))
    plc.append(mean_agreement(sector_profile(d)).mean())
print(f"{len(agree)} countries. Average Spearman correlation between a country's sector ranking and the average of the others: {agree.mean():+.2f} (placebo with shuffled sectors: {np.mean(plc):+.2f}, range {min(plc):+.2f} to {max(plc):+.2f}). Positive in {int((agree > 0).sum())} of {len(agree)} countries.")
avg = P.mean().sort_values(); cnt = P.notna().sum()
fig = px.bar(avg.reset_index(name="premium"), x="premium", y="sector", orientation="h", title="Average sector ROA premium over the country median (pp), across countries", hover_data={"premium": ":.2f"})
fig.update_layout(height=420, yaxis_title=""); fig.show()
pd.DataFrame({"mean premium (pp)": avg, "countries": cnt.reindex(avg.index), "share positive": (P > 0).sum().reindex(avg.index) / cnt.reindex(avg.index)}).round(2)

28 countries. Average Spearman correlation between a country's sector ranking and the average of the others: +0.47 (placebo with shuffled sectors: +0.00, range -0.21 to +0.15). Positive in 26 of 28 countries.


,mean premium (pp),countries,share positive
sector,,,
Financial Services,-2.380,28,0.040
Utilities,-1.320,22,0.270
Real Estate,-0.840,26,0.350
Healthcare,0.040,26,0.540
Communication Services,0.200,25,0.520
Basic Materials,0.580,28,0.790
Industrials,0.830,28,0.750
Consumer Cyclical,1.050,28,0.710
Energy,1.320,21,0.710


## 6 · Robustness
The pooled random-effects estimate for each trait in different samples. 'Same sign' = the sign of the primary estimate.

In [8]:
VARIANTS = {
 "all companies": lambda d: d,
 "excluding financials & real estate": lambda d: d[~d.sector.isin(["Financial Services", "Real Estate"])],
 "fiscal years 2023 and later": lambda d: d[d.fiscal_year >= 2023],
 "EU countries only": lambda d: d[d.iso3.isin(dim.index[dim.is_eu])],
 "non-EU countries only": lambda d: d[~d.iso3.isin(dim.index[dim.is_eu])],
 "excluding small-sample countries": lambda d: d[~d.iso3.isin(dim.index[dim.small_firm_sample])],
}
rob = {}
for h, t in TERMS.items():
    rows = []
    for v, f in VARIANTS.items():
        s = within.country_slopes(f(fy), [t]); s = s[s.term == t]
        re = within.random_effects(s.coef, s.se); sg = float((np.sign(s.coef) == np.sign(res[h]["re"]["mean"])).mean())
        rows.append({"sample": v, "pooled (pp)": re["mean"] * unit[h], "p": re["p"], "countries": re["k"], "same sign as primary (share of countries)": sg})
    other = [x for x in TERMS.values() if x != t][0]
    s2 = within.country_slopes(fy, [t, other]); s2 = s2[s2.term == t]; re2 = within.random_effects(s2.coef, s2.se)
    rows.append({"sample": f"also controlling for {other}", "pooled (pp)": re2["mean"] * unit[h], "p": re2["p"], "countries": re2["k"], "same sign as primary (share of countries)": float((np.sign(s2.coef) == np.sign(res[h]["re"]["mean"])).mean())})
    rob[h] = pd.DataFrame(rows).set_index("sample")
    print(h, t); print(rob[h].round(3).to_string()); print()

W1 equity_ratio_w
                                    pooled (pp)     p  countries  same sign as primary (share of countries)
sample                                                                                                     
all companies                             1.159 0.000         28                                      0.964
excluding financials & real estate        1.298 0.000         26                                      0.923
fiscal years 2023 and later               1.190 0.000         28                                      1.000
EU countries only                         1.186 0.000         15                                      1.000
non-EU countries only                     1.096 0.000         13                                      0.923
excluding small-sample countries          1.158 0.000         27                                      0.963
also controlling for size_rank            1.200 0.000         28                                      0.964



W2 size_rank
                                     pooled (pp)     p  countries  same sign as primary (share of countries)
sample                                                                                                      
all companies                             -0.426 0.733         28                                      0.500
excluding financials & real estate         1.237 0.455         26                                      0.423
fiscal years 2023 and later               -0.982 0.438         28                                      0.607
EU countries only                          1.946 0.254         15                                      0.400
non-EU countries only                     -3.219 0.054         13                                      0.615
excluding small-sample countries          -0.706 0.568         27                                      0.519
also controlling for equity_ratio_w        2.045 0.118         28                                      0.429



## 7 · Verdicts

In [9]:
expected = {"W1": 1, "W2": 0}
titles = {"W1": ("Equity ratio and ROA inside countries", "Within a country, companies with a higher equity ratio earn higher ROA."),
          "W2": ("Size and ROA inside countries", "Within a country, company size is related to ROA (direction not predicted).")}
lines_all = []
for h, t in TERMS.items():
    r, re = res[h], res[h]["re"]
    v = utils.verdict(re["mean"], re["p"], expected[h], r["same_sign_share"], re["k"])
    rob_same = float((np.sign(rob[h]["pooled (pp)"]) == np.sign(re["mean"])).mean())
    lines = [f"Pooled across {re['k']} countries: {re['mean'] * unit[h]:+.2f} pp ({'per +0.1 equity ratio' if h == 'W1' else 'largest vs smallest company'}; 95% CI {re['ci_low'] * unit[h]:+.2f} to {re['ci_high'] * unit[h]:+.2f}), random-effects p = {re['p']:.4f}.",
             f"Repeats: same sign in {r['same_sign_share']:.0%} of countries ({r['significant_same_sign']} significantly, {r['significant_opposite']} significantly opposite); leave-one-country-out agreement {r['loo_agreement']:.0%}; between-country spread tau = {re['tau'] * unit[h]:.2f} pp, I2 = {re['I2']:.0%}.",
             f"Robustness: pooled sign unchanged in {rob_same:.0%} of {len(rob[h])} samples.",
             f"VERDICT {h}: {v.upper()}"]
    print("\n".join(lines)); print()
    utils.save_result(h, id=h, title=titles[h][0], statement=titles[h][1], expected_sign=expected[h], effect=re["mean"] * unit[h], ci_low=re["ci_low"] * unit[h], ci_high=re["ci_high"] * unit[h],
                      effect_unit="pp ROA per +0.1 equity ratio, within country" if h == "W1" else "pp ROA, largest vs smallest company, within country",
                      p_primary=re["p"], p_method="random-effects meta-analysis across countries (Hartung-Knapp)", n=int(slopes[slopes.term == t].n.sum()), clusters=re["k"], robust_share=r["same_sign_share"],
                      replication_coef=None, replication_p=None, replication_same_sign=None, verdict=v, notebook="3.1_within_country_company_traits.ipynb", notes=lines)

Pooled across 28 countries: +1.16 pp (per +0.1 equity ratio; 95% CI +0.96 to +1.36), random-effects p = 0.0000.
Repeats: same sign in 96% of countries (22 significantly, 0 significantly opposite); leave-one-country-out agreement 96%; between-country spread tau = 0.38 pp, I2 = 58%.
Robustness: pooled sign unchanged in 100% of 7 samples.
VERDICT W1: SUPPORTED

Pooled across 28 countries: -0.43 pp (largest vs smallest company; 95% CI -2.96 to +2.11), random-effects p = 0.7327.
Repeats: same sign in 50% of countries (5 significantly, 4 significantly opposite); leave-one-country-out agreement 50%; between-country spread tau = 5.09 pp, I2 = 77%.
Robustness: pooled sign unchanged in 57% of 7 samples.
VERDICT W2: NOT SUPPORTED



### Limits
- **Association, not causation.** The equity ratio can rise because a firm was profitable (retained earnings) as much as the reverse. Within-country comparisons remove *country* confounding, not *company* confounding.
- Only listed companies, 2022–2025 (Yahoo), and 28 of 34 countries (six have too few companies for their own estimate).
- Each country's estimate rests on a few hundred companies; individual country results are noisy, which is why the random-effects pooling and the sign share are the evidence.
- Sector classification is Yahoo's and coarse (11 sectors); companies are compared against others in the same sector only through the sector control, not matched one-to-one.
- This notebook does not use shocks or exposure yet; the next step in this series (3.2) is the shift-share comparison, where firms in the same country that differ in exposure to a country-level shock are compared.

In [10]:
con.close()